<a href="https://colab.research.google.com/github/taruchit/Building-RAG-for-answering-question-from-resume/blob/main/RAG_on_Resume.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# Import the required packages
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

In [20]:
# Load the HTML as a LangChain document loader
loader = PyPDFLoader(file_path="/content/Resume_Taruchit.pdf")
resume_docs = loader.load()

In [21]:
# Inspect the first page
print(f"Total pages: {len(resume_docs)}")
print(resume_docs[0].page_content[:200])  # First 1000 characters
print(resume_docs[0].metadata)

Total pages: 2
TARUCHIT GOYAL 
Over 9.5 years of work experience in technology industry; 
Have worked with large scale distributed data platforms  
and data lake. Hands-on experience on building stable and 
efficien
{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-11-20T00:08:46+05:30', 'author': 'Taruchit Goyal', 'moddate': '2026-11-20T00:08:46+05:30', 'source': '/content/Resume_Taruchit.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


In [22]:
import os
from google.colab import userdata

# Securely retrieve the OpenAI API key from Colab Secrets
try:
    openai_api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    # Fallback to os.environ if not found in Colab Secrets
    openai_api_key = os.environ.get("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("Please set your 'OPENAI_API_KEY' in the Colab Secrets (the key icon on the left sidebar).")

# Load the models required to complete the exercise using the retrieved API key
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=openai_api_key)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=openai_api_key)

In [23]:
# Initialize RecursiveCharacterTextSplitter to make chunks of PDF text
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [24]:
# Split the resume document with text_splitter
splits = text_splitter.split_documents(resume_docs)

In [25]:
# Initialize Chroma vectorstore with documents as splits and using OpenAIEmbeddings
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

In [26]:
# Setup vectorstore as retriever
retriever = vectorstore.as_retriever()

In [27]:
# Define RAG prompt
prompt = ChatPromptTemplate.from_template("You are an assistant for question-answering tasks to hiring manager to give information about the candidate from the resume. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:")

In [28]:
# Setup the chain
rag_chain = (
    {"context": retriever , "question": RunnablePassthrough()}
    | prompt
    | llm
)

In [29]:
# Initialize query
query = "Does the candidate knows about GenAI?"

In [30]:
# Invoke the query
answer = rag_chain.invoke(query).content
print(answer)

Yes, the candidate is knowledgeable about GenAI, as evidenced by their experience with large language models (LLMs) in projects such as text translation and designing an AI agent framework. They have also completed a certification on Introduction to Large Language Models from IIT Delhi. This background indicates a solid understanding of generative AI concepts and applications.
